# Solver scaling: equivalent allocation problems, measured carefully

Run **Runtime → Run all** in Google Colab. This CPU-only notebook compares the MILP and specialized LP allocation paths on deterministic synthetic workloads. No API keys, ERP actions, or external dispatch are involved.

The transportation-style allocation model can have integral LP optima for integral supply and demand. This supports an LP fast path only inside its documented admissible regime. A relaxation bound alone is not an executable integer plan, and arbitrary rounding is not a valid substitute for feasibility checks.

These synthetic scaling measurements are not performance claims for the published warehouse dataset or your production workload. Solver timings depend on runtime, solver version, thread scheduling, and warm-up.

In [ ]:
import importlib
import importlib.util
import subprocess
import sys

needs_install = importlib.util.find_spec("assumption_ops") is None
if not needs_install:
    import assumption_ops
    needs_install = not hasattr(assumption_ops, "optimize_lp")
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "git+https://github.com/crypticsymmetry/improved-giggle.git",
    ])
    for module_name in list(sys.modules):
        if module_name == "assumption_ops" or module_name.startswith("assumption_ops."):
            del sys.modules[module_name]
    importlib.invalidate_caches()

import csv
import json
from math import isclose
from pathlib import Path
from pprint import pprint

from assumption_ops import Order, Policy, Supply, evaluate_plan, optimize, validate_plan
from assumption_ops.scaling import benchmark_solvers


## 1. Run equal-input repeated benchmarks

Both solver paths receive identical facts and weights for each workload. The benchmark warms both paths outside measured repetitions and rotates their execution order to reduce simple order effects. Three seeds and three measured repetitions per size provide repeat distributions rather than a single stopwatch reading.

Compare objective, feasibility, and status. Alternative optimal allocations can differ, so equality of allocation tuples is not a correctness criterion. The source fingerprints let you identify the exact generated workload.

In [ ]:
order_counts = (25, 100, 500)
seeds = (1729, 1730, 1731)
repeats = 3
report = benchmark_solvers(order_counts=order_counts, seeds=seeds, repeats=repeats)
pprint(report["environment"])


## 2. Inspect measured distributions and solver equivalence

A faster result is useful only if it solves the same objective and passes the same hard constraints. Inspect median timing alongside every-run status and objective diagnostics. Close medians on tiny cases can reflect fixed overhead; do not extrapolate an apparent ratio to larger or differently constrained workloads.

In [ ]:
from IPython.display import Markdown, display

headers = ["Orders", "Verified pairs", "Unverified pairs", "MILP median (s)", "LP median (s)", "Median ratio"]
lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
for row in report["summary"]:
    def number(value):
        return "unavailable" if value is None else f"{value:.6f}"
    values = [
        str(row["orders"]), str(row["verified_pairs"]), str(row["unverified_pairs"]),
        number(row["milp_median_seconds"]), number(row["lp_median_seconds"]),
        number(row["ratio_milp_to_lp_medians"]),
    ]
    lines.append("| " + " | ".join(values) + " |")
display(Markdown("\n".join(lines)))
pprint(report["methodology"])


## 3. Inspect individual runs

The complete report retains per-run timings and workload provenance. Objective differences should be evaluated with numerical tolerance, while feasibility and integral allocation requirements remain hard constraints. None of these tests requires identical tie-breaking.

In [ ]:
pairs = {}
for run in report["runs"]:
    pairs.setdefault(run["pair"], []).append(run)
    metrics = run["metrics"]
    assert metrics["allocated_units"] + metrics["unfilled_units"] == metrics["requested_units"]
    assert metrics["on_time_units"] + metrics["late_units"] == metrics["allocated_units"]
for paired_runs in pairs.values():
    assert len(paired_runs) == 2
    assert {run["solver"] for run in paired_runs} == {"lp", "milp"}
    assert len({run["input_fingerprint"] for run in paired_runs}) == 1
    assert {run["position"] for run in paired_runs} == {0, 1}
    if all(run["status"] == "optimal" for run in paired_runs):
        assert all(run["proven_equal_quality"] for run in paired_runs)
        assert isclose(
            paired_runs[0]["objective"], paired_runs[1]["objective"],
            rel_tol=1e-9, abs_tol=1e-6,
        )
assert report["synthetic"] is True
assert report["external_dispatch"] is False
print(f"Verified input pairing and accounting across {len(report['runs'])} measured runs.")
pprint(report["runs"][:6])


## 4. Explicit solver selection and conservative fallback

`solver="lp"` is restricted to its supported model and rejects a previous-plan disruption objective. `solver="auto"` uses the existing MILP path when a previous plan is supplied. The default remains `solver="milp"`.

The example below deliberately passes a previous plan. It confirms that explicit LP refuses the unsupported request and that automatic selection produces a feasible result with the same objective as explicit MILP. It does not demonstrate arbitrary rounding of fractional allocations.

In [ ]:
supplies = [Supply("stock", "A", 3, 0, 1), Supply("shipment", "A", 7, 2, 1)]
orders = [Order("urgent", "A", 5, 3, 3), Order("standard", "A", 5, 4, 1)]
policy = Policy(disruption_penalty=10)
initial = optimize(supplies, orders, policy, solver="lp")
validate_plan(supplies, orders, policy, initial)
evaluate_plan(supplies, orders, policy, initial)

try:
    optimize(supplies, orders, policy, previous=initial, solver="lp")
except ValueError as exc:
    print(f"Expected unsupported-LP rejection: {exc}")
else:
    raise AssertionError("Explicit LP accepted a previous-plan objective")

automatic = optimize(supplies, orders, policy, previous=initial, solver="auto")
explicit_milp = optimize(supplies, orders, policy, previous=initial, solver="milp")
validate_plan(supplies, orders, policy, automatic)
evaluate_plan(supplies, orders, policy, automatic, previous=initial)
assert isclose(automatic.objective, explicit_milp.objective, rel_tol=1e-9, abs_tol=1e-6)
pprint({"automatic": automatic.to_dict(), "explicit_milp": explicit_milp.to_dict()})


## 5. Export the full reproducible report

The JSON preserves environment, source fingerprints, timing distributions, and every run. CSV exposes individual run fields; structured values are JSON-encoded rather than silently discarded. Keep the JSON companion when sharing timing rows.

In [ ]:
output_directory = Path("solver_scaling_outputs")
output_directory.mkdir(exist_ok=True)
json_path = output_directory / "solver_scaling.json"
json_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
csv_path = output_directory / "solver_scaling_runs.csv"
run_rows = [
    {**run, **{f"environment_{key}": value for key, value in report["environment"].items()}}
    for run in report["runs"]
]
fieldnames = sorted({key for row in run_rows for key in row})
with csv_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=fieldnames)
    writer.writeheader()
    for row in run_rows:
        writer.writerow({
            key: json.dumps(value, sort_keys=True) if isinstance(value, (dict, list, tuple)) else value
            for key, value in row.items()
        })
print(json_path)
print(csv_path)


## 6. Change the experiment deliberately

Keep seeds, repetitions, solver versions, and input fingerprints with your results. Larger sizes may take longer and should be tested before attributing a scaling difference to the algorithm. Changes that add non-transportation constraints can invalidate the LP admissible regime and require MILP or a different formulation.

The optional call below is commented out so the default notebook remains small. For real operational comparisons, benchmark your validated data adapter separately and retain its source-semantic caveats.

In [ ]:
experiment_configuration = {
    "order_counts": order_counts,
    "seeds": seeds,
    "repeats": repeats,
    "source": "deterministic synthetic generator",
}
pprint(experiment_configuration)
# Optional larger repeat experiment:
# larger_report = benchmark_solvers(
#     order_counts=(100, 500, 1000), seeds=(1729, 1730, 1731), repeats=5
# )
